In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import os 
import torch
from torch.autograd import Variable
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torchvision import transforms
from PIL import Image
from einops import rearrange, repeat
import numpy as np
import sys

sys.path.append('..')
from utils import resize_to_max_size, preprocess_image, preprocess_greyscale_image, postp, LuminanceRemapper, GramMatrix, MaskedGramMSELoss, MaskedMSELoss, Mean_Std_MSELoss, TotalVariationAbsLoss, resize_scalar_field, receptive_resize_scalar_field, minpool_resize_scalar_field, calc_mean_std

base_dir = os.path.join(os.getcwd(), "..")
content_dir = os.path.join(base_dir, 'images/content')
style_dir = os.path.join(base_dir, "images/styles")
model_dir = os.path.join(base_dir, "SANet/models")
scalar_field_dir = os.path.join(base_dir, "images/scalar_fields")

print("pytorch version: ", torch.__version__)
print("pytorch cuda version: ", torch.version.cuda)
print("cuda is available: ", torch.cuda.is_available())
print("number of available cpus: ", os.cpu_count())
print("current working directory: ", os.getcwd())

In [ ]:
#vgg definition that conveniently let's you grab the outputs from any layer
class VGG(nn.Module):
    def __init__(self, pool='max'):
        super(VGG, self).__init__()
        #vgg modules
        self.conv1_1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.conv3_1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv3_2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv3_3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv3_4 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv4_1 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.conv4_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv4_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv4_4 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_1 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_4 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        if pool == 'max':
            self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool5 = nn.MaxPool2d(kernel_size=2, stride=2)
        elif pool == 'avg':
            self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool3 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool4 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool5 = nn.AvgPool2d(kernel_size=2, stride=2)
            
    def forward(self, x, out_keys):
        out = {}
        out['r11'] = F.relu(self.conv1_1(x))
        out['r12'] = F.relu(self.conv1_2(out['r11']))
        out['p1'] = self.pool1(out['r12'])
        out['r21'] = F.relu(self.conv2_1(out['p1']))
        out['r22'] = F.relu(self.conv2_2(out['r21']))
        out['p2'] = self.pool2(out['r22'])
        out['r31'] = F.relu(self.conv3_1(out['p2']))
        out['r32'] = F.relu(self.conv3_2(out['r31']))
        out['r33'] = F.relu(self.conv3_3(out['r32']))
        out['r34'] = F.relu(self.conv3_4(out['r33']))
        out['p3'] = self.pool3(out['r34'])
        out['r41'] = F.relu(self.conv4_1(out['p3']))
        out['r42'] = F.relu(self.conv4_2(out['r41']))
        out['r43'] = F.relu(self.conv4_3(out['r42']))
        out['r44'] = F.relu(self.conv4_4(out['r43']))
        out['p4'] = self.pool4(out['r44'])
        out['r51'] = F.relu(self.conv5_1(out['p4']))
        out['r52'] = F.relu(self.conv5_2(out['r51']))
        out['r53'] = F.relu(self.conv5_3(out['r52']))
        out['r54'] = F.relu(self.conv5_4(out['r53']))
        out['p5'] = self.pool5(out['r54'])
        return [out[key] for key in out_keys]

In [ ]:
#get network
vgg = VGG()
vgg_dict_sanet = torch.load(os.path.join(model_dir, 'vgg_normalised.pth'), weights_only=True)
# translate state dict from sanet for easier access
translation_dict = {
    "2.weight" : "conv1_1.weight",
    "2.bias" : "conv1_1.bias",
    "5.weight" : "conv1_2.weight",
    "5.bias" : "conv1_2.bias",
    "9.weight" : "conv2_1.weight",
    "9.bias" : "conv2_1.bias",
    "12.weight" : "conv2_2.weight",
    "12.bias" : "conv2_2.bias",
    "16.weight" : "conv3_1.weight",
    "16.bias" : "conv3_1.bias",
    "19.weight" : "conv3_2.weight",
    "19.bias" : "conv3_2.bias",
    "22.weight" : "conv3_3.weight",
    "22.bias" : "conv3_3.bias",
    "25.weight" : "conv3_4.weight",
    "25.bias" : "conv3_4.bias",
    "29.weight" : "conv4_1.weight",
    "29.bias" : "conv4_1.bias",
    "32.weight" : "conv4_2.weight",
    "32.bias" : "conv4_2.bias",
    "35.weight" : "conv4_3.weight",
    "35.bias" : "conv4_3.bias",
    "38.weight" : "conv4_4.weight",
    "38.bias" : "conv4_4.bias",
    "42.weight" : "conv5_1.weight",
    "42.bias" : "conv5_1.bias",
    "45.weight" : "conv5_2.weight",
    "45.bias" : "conv5_2.bias",
    "48.weight" : "conv5_3.weight",
    "48.bias" : "conv5_3.bias",
    "51.weight" : "conv5_4.weight",
    "51.bias" : "conv5_4.bias"
}
vgg_dict = dict((translation_dict.get(key, key), value) for (key, value) in vgg_dict_sanet.items())
#NOTE: the vgg from SANET has a 1x1 conv. layer to do what is done here with preprocessing (prep function)
# -> remove keys, 0.weight and 0.bias
del vgg_dict["0.weight"]
del vgg_dict["0.bias"]
vgg.load_state_dict(vgg_dict)
for param in vgg.parameters():
    param.requires_grad = False
if torch.cuda.is_available():
    vgg.cuda()

## Choose content image, style image, scalar field and optimization parameters

In [ ]:
style_img_names = ['candy.jpg']
content_img_names = ['471.jpg']
scalar_field_name = "gradient_for_471.jpg"

max_style_size = 512
max_content_size = 1024

mask = "scalar_field" # can use "none", "scalar_field", "binary" and "gradient"

downsampling_type = "minpool" # "minpool", "bilinear" and "receptive"
downsampling_use_relu = False # whether to use relu layers in the downsampling network

mask_content_loss = True

#NOTE: weight ratio between gram and mse+std loss
weight_mean_std = 1.0
weight_gram = 1.0 - weight_mean_std

max_iter = 500 # optimization steps

remap_colors = True

use_content_mask = False #NOTE: only use this region for luminance remapping (e.g. when we transfer a texture map with non-rectangular boundary)
content_mask_path = os.path.join(base_dir, "<content_mask_path>")

style_imgs = [Image.open(os.path.join(style_dir, name)) for name in style_img_names]
content_imgs = [Image.open(os.path.join(content_dir, name)) for name in content_img_names]
scalar_field_img = Image.open(os.path.join(scalar_field_dir, scalar_field_name)) 

if use_content_mask:
    content_mask_img = Image.open(content_mask_path) 
    content_mask_img = resize_to_max_size(content_mask_img.convert("L"), max_content_size)
    content_mask = np.array(content_mask_img).astype(np.bool) # mask is binary
else:
    content_mask = None

# color remapping
if remap_colors:
    color_remapper = LuminanceRemapper(resize_to_max_size(content_imgs[0], max_content_size), content_mask = content_mask)
    style_imgs = [color_remapper.shift(style_img) for style_img in style_imgs]

In [ ]:
# resize content image to given size
preprocess_style = preprocess_image(max_style_size, multiple_of_16 = False) #NOTE: need multiple of 16 when masking style
preprocess_content = preprocess_image(max_content_size)

style_imgs_torch = [preprocess_style(img) for img in style_imgs]
content_imgs_torch = [preprocess_content(img) for img in content_imgs]

init_img_torch = content_imgs_torch[0].data.clone()

if torch.cuda.is_available():
    style_imgs_torch = [Variable(img.unsqueeze(0).cuda()) for img in style_imgs_torch]
    content_imgs_torch = [Variable(img.unsqueeze(0).cuda()) for img in content_imgs_torch]
    init_img_torch = init_img_torch.unsqueeze(0).cuda()
else:
    style_imgs_torch = [Variable(img.unsqueeze(0)) for img in style_imgs_torch]
    content_imgs_torch = [Variable(img.unsqueeze(0)) for img in content_imgs_torch]
    init_img_torch = init_img_torch.unsqueeze(0)
    print("using cpu as cuda is not available")
content_image = content_imgs_torch[0]

# for img in style_imgs_torch:
#     print("style image size: ", img.shape)
# print("content image size: ", content_image.shape)

# opt_img = Variable(torch.randn(content_image.size()).type_as(content_image.data), requires_grad=True) #NOTE: random init
opt_img = Variable(init_img_torch, requires_grad=True)

## Process / generate scalar field

In [ ]:
scalar_fields = []

# no mask
if mask == "none":
    scalar_field = torch.ones(1, 1, content_image.shape[-2], content_image.shape[-1])

#NOTE: binary mask
elif mask == "binary":
    scalar_field = 1e-9 * torch.ones(1, 1, content_image.shape[-2], content_image.shape[-1])
    scalar_field[:, :, :, content_image.shape[3]//2:] = 1.0

#NOTE: gradient mask
elif mask == "gradient":
    scalar_field = torch.linspace(1.0, 1e-6, content_image.shape[-1])
    # scalar_field = torch.linspace(1e-6, 1.0, content_image.shape[-1])
    scalar_field = repeat(scalar_field, "w -> 1 1 h w", h = content_image.shape[-2])

#NOTE: scalar field mask
elif mask == "scalar_field":
    preprocess_greyscale_mask = preprocess_greyscale_image(max_content_size)
    scalar_field = preprocess_greyscale_mask(scalar_field_img.convert("L"))
    # normalization of scalar field
    if use_content_mask:
        content_mask_torch = rearrange(torch.from_numpy(content_mask), "h w -> 1 h w")
        scalar_field_max = scalar_field[content_mask_torch].max()
        scalar_field_min = scalar_field[content_mask_torch].min()
        scalar_field = (scalar_field - scalar_field_min) / (scalar_field_max - scalar_field_min)
    else:
        scalar_field = (scalar_field - scalar_field.min()) / (scalar_field.max() - scalar_field.min())
    scalar_field = torch.clip(scalar_field, 1e-6, 1.0) 
    scalar_field = rearrange(scalar_field, "c h w -> 1 c h w")

scalar_fields.append(scalar_field)

scalar_field_imgs = [transforms.functional.to_pil_image(repeat(rearrange(scalar_field, "1 1 h w -> 1 h w"), "1 h w -> c h w", c=3), mode = "RGB") for scalar_field in scalar_fields]
# move scalar field to gpu
scalar_fields = [scalar_field.cuda() for scalar_field in scalar_fields]

In [ ]:
#display images
for img in style_imgs:
    plt.imshow(img);plt.show()
for img in content_imgs:
    plt.imshow(img);plt.show()
for img in scalar_field_imgs:
    plt.imshow(img);plt.show()

## Set up loss function, optimization targets and downsample scalar field

In [ ]:
#d efine layers, loss functions, weights and compute optimization targets
style_layers = ['r11', 'r21', 'r31', 'r41', 'r51']
content_layers = ['r41']
# total_variation_layers = ['r11']
loss_layers = style_layers + content_layers # + total_variation_layers

style_loss_fns_gram = [MaskedGramMSELoss()] * len(style_layers) * len(style_imgs)
style_loss_fns_mean_std = [Mean_Std_MSELoss()] * len(style_layers) * len(style_imgs)

if mask_content_loss:
    content_loss_fns = [MaskedMSELoss()] * len(content_layers) * len(style_imgs)
else:
    content_loss_fns = [nn.MSELoss()] * len(content_layers) * len(style_imgs)

# total_variation_loss_fns = [TotalVariationAbsLoss] * len(total_variation_layers)

if torch.cuda.is_available():
    style_loss_fns_gram = [loss_fn.cuda() for loss_fn in style_loss_fns_gram]
    style_loss_fns_mean_std = [loss_fn.cuda() for loss_fn in style_loss_fns_mean_std]
    content_loss_fns = [loss_fn.cuda() for loss_fn in content_loss_fns]
    
# these are good weights settings:
style_weights = [1e3 for n in [64,128,256,512,512]] * len(style_imgs)
content_weights = [1e0] * len(style_imgs)
total_variation_weights = [1e1]

#NOTE downsample scalar field
resized_scalar_fields_style_layers = []
resized_scalar_fields_content_layers = []
for scalar_field in scalar_fields:
    if downsampling_type == "minpool":
        resized_scalar_fields = [A.detach() for A in minpool_resize_scalar_field(scalar_field, loss_layers, use_relu = downsampling_use_relu)]
    elif downsampling_type == "receptive":
        resized_scalar_fields = [A.detach() for A in receptive_resize_scalar_field(scalar_field, loss_layers, use_relu = downsampling_use_relu)]
    elif downsampling_type == "bilinear":
        resized_scalar_fields = [A.detach() for A in resize_scalar_field(scalar_field, loss_layers)]
    else:
        raise ValueError(f"Unknown downsampling type = {downsampling_type}, use one of ['minpool', 'receptive', 'bilinear']")
    resized_scalar_fields_style_layers.extend(resized_scalar_fields[:len(style_layers)])
    resized_scalar_fields_content_layers.extend(resized_scalar_fields[len(style_layers):])

# compute optimization targets
style_features = []
for style_img in style_imgs_torch:
    style_features.extend(vgg(style_img, style_layers))
# style targets
style_targets_gram = [GramMatrix()(A).detach() for A in style_features]
style_targets_mean_std = [calc_mean_std(A) for A in style_features]
style_targets_mean_std = [(A[0].detach(), A[1].detach()) for A in style_targets_mean_std]
# duplicate style and content layers to match the number of style images
style_layers *= len(style_imgs)
content_layers *= len(style_imgs)
loss_layers = style_layers + content_layers
content_targets = [A.detach() for A in vgg(content_image, content_layers)] 

In [ ]:
# plot resized scalar fields
for tensor in resized_scalar_fields_style_layers:
    img = tensor[0,0].detach().cpu()
    img = transforms.functional.to_pil_image(repeat(img, "h w -> c h w", c=3), mode = "RGB")
    plt.imshow(img);plt.show()

## Run style transfer

In [ ]:
# run style transfer
style_names = [img_name.split(".")[0] for img_name in style_img_names]
content_names = [img_name.split(".")[0] for img_name in content_img_names]
scalar_field_name = scalar_field_name.split(".")[0]
output_name = "images/output/gatysnst/"
if mask_content_loss:
    output_name += "mask_style+content_loss"
else:
    output_name += "mask_style_loss"
if mask == "scalar_field":
    output_name += f"_scalar_field={scalar_field_name}"
elif mask != "none": 
    output_name += "_" + mask 
output_name += f"_styles={style_names}_content={content_names}_size={max_content_size}"
if remap_colors:
    output_name += "_remap_colors"
if mask != "none":
    if downsampling_type == "bilinear":
        output_name += "_bilinear_downsampling"
    elif downsampling_type == "receptive":
        output_name += "_receptive_downsampling"
    if downsampling_use_relu:
        output_name += "_with_relu"
show_iter = 10
output_name += f"_steps={max_iter}"
optimizer = optim.LBFGS([opt_img]);
n_iter=[0]

while n_iter[0] <= max_iter:

    def closure():
        optimizer.zero_grad()
        out = vgg(opt_img, loss_layers) # list of features from loss_layers
        out_features_for_style_loss = out[:len(style_layers)]
        out_features_for_content_loss = out[len(style_layers): len(style_layers) + len(content_layers)]
        # out_features_for_total_variation_loss = out[len(style_layers) + len(content_layers):]
        style_losses_mean_std = [style_weights[idx] * style_loss_fns_mean_std[idx](A, style_targets_mean_std[idx], resized_scalar_fields_style_layers[idx]) for idx,A in enumerate(out_features_for_style_loss)]
        style_losses_gram = [style_weights[idx] * style_loss_fns_gram[idx](A, style_targets_gram[idx], resized_scalar_fields_style_layers[idx]) for idx,A in enumerate(out_features_for_style_loss)]
        if mask_content_loss:
            content_losses = [content_weights[idx] * content_loss_fns[idx](A, content_targets[idx], resized_scalar_fields_content_layers[idx]) for idx,A in enumerate(out_features_for_content_loss)]
        else:
            content_losses = [content_weights[idx] * content_loss_fns[idx](A, content_targets[idx]) for idx,A in enumerate(out_features_for_content_loss)] # no mask
        # total_variation_losses = [total_variation_weights[idx] * total_variation_loss_fns[idx](A) for idx, A in enumerate(out_features_for_total_variation_loss)]
        style_loss = weight_mean_std * torch.stack(style_losses_mean_std).sum() + weight_gram * torch.stack(style_losses_gram).sum()
        content_loss = torch.stack(content_losses).sum()
        # total_variation_loss = torch.stack(total_variation_losses).sum()
        loss = (style_loss + content_loss) / len(style_imgs) # + total_variation_loss
        loss.backward()

        #NOTE: gradient masking, can help with areas where no change should be made, but generally not necessary
        # opt_img.grad *= scalar_field

        n_iter[0]+=1
        if n_iter[0]%show_iter == (show_iter-1):
            print('Iteration: %d, style_loss: %f, content_loss: %f'%(n_iter[0]+1, style_loss.item(), content_loss.item()))
        return loss
    
    optimizer.step(closure)
    
#display result
out_img = postp(opt_img.data[0].cpu().squeeze())

# color remapping
if remap_colors:
    out_img = color_remapper.unshift(out_img)

plt.imshow(out_img)
plt.gcf().set_size_inches(10,10)

# save image
out_img.save(os.path.join(base_dir, output_name + ".png"), "PNG")

### Image space interpolation

In [ ]:
assert mask == "no", "'mask' needs to be set to 'no' for this to work."
# interpolate in image space
stylized_img = opt_img.data[0].cpu().squeeze()
content_img = content_imgs_torch[0].cpu().squeeze()

interpolation_mask = "gradient"

#NOTE: binary mask
if interpolation_mask == "binary":
    scalar_field = 1e-9 * torch.ones(1, content_image.shape[-2], content_image.shape[-1])
    scalar_field[:, :, content_image.shape[3]//2:] = 1.0

#NOTE: gradient mask
elif interpolation_mask == "gradient":
    scalar_field = torch.linspace(1.0, 1e-6, content_image.shape[-1])
    # scalar_field = torch.linspace(1e-6, 1.0, content_image.shape[-1])
    scalar_field = repeat(scalar_field, "w -> 1 h w", h = content_image.shape[-2])

#NOTE: greyscale mask
elif interpolation_mask == "scalar_field":
    preprocess_greyscale_mask = preprocess_greyscale_image(max_content_size, multiple_of_16=True)
    scalar_field = preprocess_greyscale_mask(scalar_field_img.convert("L"))
    # fix normalization of scalar field
    if use_content_mask:
        content_mask_torch = rearrange(torch.from_numpy(content_mask), "h w -> 1 h w")
        scalar_field_max = scalar_field[content_mask_torch].max()
        scalar_field_min = scalar_field[content_mask_torch].min()
        scalar_field = (scalar_field - scalar_field_min) / (scalar_field_max - scalar_field_min)
    else:
        scalar_field = (scalar_field - scalar_field.min()) / (scalar_field.max() - scalar_field.min())
    scalar_field = torch.clip(scalar_field, 1e-6, 1.0) 

# color remapping
if remap_colors:
    preprocess_raw_content = preprocess_image(max(stylized_img.shape[1], stylized_img.shape[2]), True)
    out_img = color_remapper.unshift(postp(stylized_img))
    stylized_img = preprocess_raw_content(out_img)

# interpolated_img = postp(stylized_img)
interpolated_img = postp(scalar_field * stylized_img + (1.0 - scalar_field) * content_img)

plt.imshow(interpolated_img)
plt.gcf().set_size_inches(10,10)

# save image
interpolated_img.save(os.path.join(base_dir, output_name + "_image_space_interpolation.png"), "PNG")